# Feature Analysis — IC/IR, Correlation Matrix, and Importance Ranking
Computes Information Coefficient (IC), Information Ratio (IR), and feature
correlation analysis for ML trading features.

In [ ]:
!pip install -q pandas numpy matplotlib scipy yfinance scikit-learn seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#131722',
    'axes.facecolor':   '#1e2433',
    'axes.edgecolor':   '#ffffff22',
    'text.color':       '#cccccc',
    'axes.labelcolor':  '#888888',
    'xtick.color':      '#888888',
    'ytick.color':      '#888888',
    'grid.color':       '#ffffff0d',
})

SYMBOL = 'BTC-USD'
print('Feature Analysis — Bloomberg dark theme')

In [ ]:
import yfinance as yf

df = yf.download(SYMBOL, start='2021-01-01', interval='1d', progress=False).dropna()
close = df['Close'].squeeze()
high  = df['High'].squeeze()
low   = df['Low'].squeeze()
vol   = df['Volume'].squeeze()

def rsi(s, n=14):
    d = s.diff(); g = d.clip(lower=0); l = -d.clip(upper=0)
    return 100-100/(1+g.ewm(com=n-1,min_periods=n).mean()/(l.ewm(com=n-1,min_periods=n).mean()+1e-9))

# Build features
feats = pd.DataFrame(index=df.index)
feats['ret_1d']         = close.pct_change()
feats['ret_5d']         = close.pct_change(5)
feats['ret_20d']        = close.pct_change(20)
feats['mom_10']         = close / close.shift(10) - 1
feats['rsi_14']         = rsi(close)
ema12                   = close.ewm(span=12).mean()
ema26                   = close.ewm(span=26).mean()
feats['macd']           = ema12 - ema26
feats['macd_signal']    = feats['macd'].ewm(span=9).mean()
bb_mid                  = close.rolling(20).mean()
bb_std                  = close.rolling(20).std()
feats['bb_pct']         = (close - bb_mid) / (bb_std * 2 + 1e-9)
feats['bb_width']       = bb_std * 2 / (bb_mid + 1e-9)
tr                      = pd.concat([high-low,(high-close.shift()).abs(),(low-close.shift()).abs()],axis=1).max(axis=1)
feats['atr_pct']        = tr.rolling(14).mean() / (close + 1e-9)
feats['vol_ratio_5d']   = vol / (vol.rolling(5).mean() + 1e-9)
feats['vol_ratio_20d']  = vol / (vol.rolling(20).mean() + 1e-9)
feats['realised_vol']   = feats['ret_1d'].rolling(20).std() * np.sqrt(252)

# Forward returns for IC calculation (1d, 5d, 20d)
feats['fwd_1d']  = close.pct_change().shift(-1)
feats['fwd_5d']  = close.pct_change(5).shift(-5)
feats['fwd_20d'] = close.pct_change(20).shift(-20)

feats = feats.dropna()
FEATURE_COLS = ['ret_1d','ret_5d','ret_20d','mom_10','rsi_14','macd','macd_signal',
                'bb_pct','bb_width','atr_pct','vol_ratio_5d','vol_ratio_20d','realised_vol']

print(f'Samples: {len(feats)} | Features: {len(FEATURE_COLS)}')
feats[FEATURE_COLS].describe().round(4)

In [ ]:
# ── IC / IR Analysis ─────────────────────────────────────────────────────────
# IC = Spearman rank correlation of feature with forward return
# IR = IC.mean() / IC.std()  (rolling window)

WINDOW = 60   # rolling window for IC

results = []
for feat in FEATURE_COLS:
    for fwd_col, horizon in [('fwd_1d', '1D'), ('fwd_5d', '5D'), ('fwd_20d', '20D')]:
        valid = feats[[feat, fwd_col]].dropna()
        if len(valid) < 30:
            continue

        # Full-sample IC
        ic_full, _ = stats.spearmanr(valid[feat], valid[fwd_col])

        # Rolling IC
        rolling_ics = []
        for i in range(WINDOW, len(valid)):
            sub = valid.iloc[i-WINDOW:i]
            ic, _ = stats.spearmanr(sub[feat], sub[fwd_col])
            rolling_ics.append(ic)
        rolling_ics = np.array(rolling_ics)

        ir = rolling_ics.mean() / (rolling_ics.std() + 1e-9)
        results.append({
            'feature':  feat,
            'horizon':  horizon,
            'ic_full':  round(ic_full, 4),
            'ic_mean':  round(rolling_ics.mean(), 4),
            'ic_std':   round(rolling_ics.std(), 4),
            'ir':       round(ir, 3),
        })

ic_df = pd.DataFrame(results)
print('IC/IR Summary (1D horizon):')
ic_1d = ic_df[ic_df['horizon'] == '1D'].sort_values('ir', ascending=False)
print(ic_1d.to_string(index=False))

In [ ]:
# ── Feature Correlation Matrix ────────────────────────────────────────────────
corr = feats[FEATURE_COLS].corr(method='spearman')

# Custom diverging colormap: red → white → green
cmap = mcolors.LinearSegmentedColormap.from_list(
    'rg_diverging',
    ['#ff1744', '#441122', '#1e2433', '#113322', '#00c853'],
    N=256
)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(FEATURE_COLS)))
ax.set_yticks(range(len(FEATURE_COLS)))
ax.set_xticklabels(FEATURE_COLS, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(FEATURE_COLS, fontsize=8)

# Annotate cells
for i in range(len(FEATURE_COLS)):
    for j in range(len(FEATURE_COLS)):
        val = corr.values[i, j]
        color = 'white' if abs(val) > 0.5 else '#888888'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6, color=color)

plt.colorbar(im, ax=ax, shrink=0.8, label='Spearman Correlation')
ax.set_title('Feature Correlation Matrix (Spearman)', fontsize=12, color='white', pad=12)
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to feature_correlation.png')

In [ ]:
# ── Feature Importance Ranking by |IC| ───────────────────────────────────────
rank_df = ic_df.groupby('feature').agg(
    mean_abs_ic = ('ic_full', lambda x: x.abs().mean()),
    mean_ir     = ('ir', 'mean'),
).sort_values('mean_abs_ic', ascending=True).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
colors  = ['#00c853' if v > 0 else '#ff1744' for v in rank_df['mean_ir']]
bars    = ax.barh(rank_df['feature'], rank_df['mean_abs_ic'], color='#2979ff', alpha=0.8)

# Overlay IR as text
for i, (bar, ir_val) in enumerate(zip(bars, rank_df['mean_ir'])):
    ax.text(
        bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
        f'IR={ir_val:.2f}',
        va='center', ha='left', fontsize=7,
        color='#00c853' if ir_val > 0 else '#ff1744'
    )

ax.set_xlabel('Mean |IC| across horizons', fontsize=9)
ax.set_title('Feature Ranking by Information Coefficient', fontsize=11, color='white', pad=10)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance_ic.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to feature_importance_ic.png')
print('\nTop 5 features by mean |IC|:')
print(rank_df.tail(5)[['feature', 'mean_abs_ic', 'mean_ir']].iloc[::-1].to_string(index=False))